In [7]:
import pickle
import geopandas as gpd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report

# Load the feature matrix
wards = gpd.read_file("../../Data/floods.gpkg")

# Drop geometry and metadata for modelling — keep only numeric features and target
feature_cols = [
    'pop2009',
    'rain_cumulative_mm',
    'rain_max_daily_mm',
    'rain_preflood_7d_mm',
    'elevation_mean_m',
    'elevation_min_m',
    'slope_mean_deg'
]

X = wards[feature_cols]
y = wards['flooded']

print(f"Feature matrix shape : {X.shape}")
print(f"Class distribution   :\n{y.value_counts(normalize=True)}")
print(f"\nMissing values:\n{X.isnull().sum()}")

Feature matrix shape : (1450, 7)
Class distribution   :
flooded
0    0.788276
1    0.211724
Name: proportion, dtype: float64

Missing values:
pop2009                0
rain_cumulative_mm     0
rain_max_daily_mm      0
rain_preflood_7d_mm    0
elevation_mean_m       0
elevation_min_m        0
slope_mean_deg         0
dtype: int64


In [8]:
# Perform train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train.head()

,pop2009,rain_cumulative_mm,rain_max_daily_mm,rain_preflood_7d_mm,elevation_mean_m,elevation_min_m,slope_mean_deg
1112,52777.0,0.000000,0.000000,0.000000,34.313187,1.0,4.642516
1298,46351.0,0.000000,0.000000,0.000000,1532.464098,1511.0,1.348357
836,55406.0,371.738520,37.046970,62.983926,1762.939130,1701.0,2.628858
1115,18806.0,0.000000,0.000000,0.000000,18.622155,1.0,1.981015
305,30960.0,138.300461,25.247042,11.300949,1129.965852,853.0,3.620829


In [9]:
# Build a simple XGBoost model
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)

# Train the model
model.fit(X_train, y_train)

# Predict on the test set
y_pred = model.predict(X_test)

# Evaluate the model
print("Classification Report:")
print(classification_report(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.95      0.92       229
           1       0.75      0.59      0.66        61

    accuracy                           0.87       290
   macro avg       0.82      0.77      0.79       290
weighted avg       0.87      0.87      0.87       290



In [10]:
# Handle class imbalance using SMOTE
from imblearn.over_sampling import SMOTE

# Instantiate SMOTE
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# Train the model on the SMOTE data
model.fit(X_train_smote, y_train_smote)

# Predict on the test set
y_pred_smote = model.predict(X_test)

# Evaluate the model
print("Classification Report after SMOTE:")
print(classification_report(y_test, y_pred_smote))


Classification Report after SMOTE:
              precision    recall  f1-score   support

           0       0.92      0.86      0.89       229
           1       0.57      0.72      0.64        61

    accuracy                           0.83       290
   macro avg       0.75      0.79      0.76       290
weighted avg       0.85      0.83      0.83       290



In [11]:
# Use GridSearchCV to find the best hyperparameters
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1]
}

grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=3, scoring='f1', n_jobs=-1)

grid_search.fit(X_train_smote, y_train_smote)
print(f"Best parameters: {grid_search.best_params_}")

# Train the model with the best parameters
best_model = grid_search.best_estimator_

best_model.fit(X_train_smote, y_train_smote)

# Predict on the test set
y_pred_best = best_model.predict(X_test)

# Evaluate the model
print("Classification Report with Best Parameters:")
print(classification_report(y_test, y_pred_best))


/home/ngundo_larry/anaconda3/envs/learn-env/lib/python3.11/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/ngundo_larry/anaconda3/envs/learn-env/lib/python3.11/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/home/ngundo_larry/anaconda3/envs/learn-env/lib/python3.11/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2

Best parameters: {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 200}
Classification Report with Best Parameters:
              precision    recall  f1-score   support

           0       0.92      0.87      0.89       229
           1       0.60      0.70      0.65        61

    accuracy                           0.84       290
   macro avg       0.76      0.79      0.77       290
weighted avg       0.85      0.84      0.84       290



Saving the model:

In [12]:
with open('../best_xg_boost_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

print("Best XGBoost model saved")

Best XGBoost model saved
